# Experiment 1 — Entropy Profile of Full Wake

## Purpose

Compute per-token **Shannon entropy** of the residual-stream activations across
pages 3–628 of Finnegans Wake (or a representative excerpt) and plot the
resulting entropy *waveform*.

High-entropy tokens correspond to positions of maximum multilingual density,
polysemy, or portmanteau condensation — the natural targets for E2's
superposition discrimination.

**Spec reference:** Section 6, Experiment E1.

### What we compute

For each token at position *i* in the sequence:

$$H_i = -\sum_j p_j \log p_j$$

where $p_j$ is the softmax probability of the $j$-th residual-stream dimension.

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import sys
import os
import json
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import plotly.graph_objects as go
import plotly.express as px

ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports OK.")

In [ ]:
# ---------------------------------------------------------------------------
# Load corpus — pages 3–10 as example
# ---------------------------------------------------------------------------
import torch

HAS_GPU = torch.cuda.is_available()
MOCK_MODE = not HAS_GPU
print(f"GPU available: {HAS_GPU} | Mock mode: {MOCK_MODE}")

# Sample FW passages (pp. 3-10 excerpt)
CORPUS_PASSAGES = [
    {"page": 3, "line": 1,  "text": "riverrun, past Eve and Adam's, from swerve of shore to bend of bay"},
    {"page": 3, "line": 2,  "text": "brings us by a commodius vicus of recirculation back to Howth Castle and Environs."},
    {"page": 3, "line": 3,  "text": "Sir Tristram, violer d'amores, fr'over the short sea, had passen-core rearrived"},
    {"page": 3, "line": 4,  "text": "from North Armorica on this side the scraggy isthmus of Europe Minor"},
    {"page": 3, "line": 5,  "text": "to wielderfight his penisolate war: nor had topsawyer's rocks by the stream Oconee"},
    {"page": 3, "line": 6,  "text": "exaggerated themselse to Laurens County's gorgios while they went doublin their mumper"},
    {"page": 3, "line": 7,  "text": "all the time: nor avoice from afire bellowsed mishe mishe to tauftauf thuartpeatrick"},
    {"page": 3, "line": 8,  "text": "not yet, though venissoon after, had a kidscad buttended a bland old isaac"},
    {"page": 3, "line": 9,  "text": "not yet, though all's fair in vanessy, were sosie sesthers wroth with twone nathandjoe."},
    {"page": 3, "line": 10, "text": "Rot a peck of pa's malt had Jhem or Shen brewed by arclight and rory end"},
    {"page": 4, "line": 1,  "text": "to the regginbrow was to be seen ringsome on the aquaface."},
    {"page": 4, "line": 2,  "text": "The fall (bababadalgharaghtakamminarronnkonnbronntonner-ronntuonnthunntrovarrhounawnskawntoohoohoordenenthurnuk!)"},
    {"page": 4, "line": 3,  "text": "of a once wallstrait oldparr is retaled early in bed and later on life down through all christian minstrelsy."},
    {"page": 5, "line": 1,  "text": "The great fall of the offwall entailed at such short notice the pftjschute of Finnegan,"},
    {"page": 5, "line": 2,  "text": "erse solid man, that the humptyhillhead of humself prumptly sends an unquiring one well to the west"},
    {"page": 6, "line": 1,  "text": "in quest of his tumptytumtoes: and their upturnpikepointandplace is at the knock out"},
    {"page": 7, "line": 1,  "text": "in the park where oranges have been laid to rust upon the green"},
    {"page": 8, "line": 1,  "text": "since devlins first loved livvy."},
    {"page": 8, "line": 2,  "text": "What clashes here of wills gen wonts, oystrygods gaggin fishy-gods!"},
    {"page": 9, "line": 1,  "text": "Bryne and Sygne sighed and cried after them."},
    {"page": 10, "line": 1, "text": "The fall of the wallstrait oldparr's wallstrait: his fall was the fall"},
]

print(f"Loaded {len(CORPUS_PASSAGES)} passages from FW pp. 3-10.")

In [ ]:
# ---------------------------------------------------------------------------
# Run model — collect per-token entropy
# ---------------------------------------------------------------------------
if not MOCK_MODE:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL_NAME = os.getenv("MODEL_NAME", "EleutherAI/gpt-j-6B")
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, output_hidden_states=True,
        torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()

def compute_token_entropies(text: str, layer: int = 16) -> List[Dict[str, Any]]:
    """Return per-token entropy dicts for *text*."""
    if MOCK_MODE:
        words = text.split()
        rng = np.random.default_rng(abs(hash(text)) % (2**32))
        # Thunder-words and portmanteaux get elevated entropy
        token_data = []
        for i, w in enumerate(words):
            base_entropy = rng.exponential(scale=5.0)
            # Boost entropy for long words (likely portmanteaux)
            if len(w) > 10:
                base_entropy *= 2.5
            token_data.append({"token": w, "entropy": float(base_entropy)})
        return token_data
    else:
        enc = hf_tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
        hidden = out.hidden_states[layer].squeeze(0).cpu().float().numpy()
        tokens_ids = enc["input_ids"].squeeze(0).tolist()
        token_strs = [hf_tokenizer.decode([t]) for t in tokens_ids]
        token_data = []
        for i, (tok, vec) in enumerate(zip(token_strs, hidden)):
            exp_v = np.exp(vec - vec.max())
            probs = exp_v / (exp_v.sum() + 1e-12)
            h = float(-np.sum(probs * np.log(probs + 1e-12)))
            token_data.append({"token": tok.strip(), "entropy": h})
        return token_data

# Collect all token entropies across the corpus
all_tokens = []
for passage in CORPUS_PASSAGES:
    token_data = compute_token_entropies(passage["text"])
    for td in token_data:
        all_tokens.append({
            "page": passage["page"],
            "line": passage["line"],
            "token": td["token"],
            "entropy": td["entropy"],
        })

print(f"Collected {len(all_tokens)} token entropy measurements.")

In [ ]:
# ---------------------------------------------------------------------------
# Plot entropy waveform with Plotly
# ---------------------------------------------------------------------------
positions = list(range(len(all_tokens)))
entropies = [t["entropy"] for t in all_tokens]
tokens_str = [t["token"] for t in all_tokens]
hover_text = [
    f"Token: {t['token']}<br>Page: {t['page']}, Line: {t['line']}<br>Entropy: {t['entropy']:.3f}"
    for t in all_tokens
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=positions,
    y=entropies,
    mode="lines+markers",
    marker=dict(
        size=4,
        color=entropies,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Entropy (nats)"),
    ),
    line=dict(width=1, color="rgba(100,100,200,0.5)"),
    text=hover_text,
    hoverinfo="text",
    name="Token entropy",
))

# Mark top-entropy tokens
threshold = np.percentile(entropies, 90)
top_positions = [i for i, e in enumerate(entropies) if e >= threshold]
top_entropies = [entropies[i] for i in top_positions]
top_tokens_str = [tokens_str[i] for i in top_positions]

fig.add_trace(go.Scatter(
    x=top_positions,
    y=top_entropies,
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=[t if len(t) < 20 else t[:18]+"…" for t in top_tokens_str],
    textposition="top center",
    name="Top 10% entropy",
))

fig.update_layout(
    title="E1 — Entropy Waveform of Finnegans Wake (pp. 3-10 excerpt)",
    xaxis_title="Token position",
    yaxis_title="Residual entropy (nats)",
    height=500,
    hovermode="x unified",
    template="plotly_dark",
)

fig.write_html(OUTPUT_DIR / "E1_entropy_waveform.html")
fig.show()
print(f"Waveform saved to outputs/E1_entropy_waveform.html")

In [ ]:
# ---------------------------------------------------------------------------
# Identify top-10 highest entropy tokens/passages
# ---------------------------------------------------------------------------
sorted_tokens = sorted(all_tokens, key=lambda t: t["entropy"], reverse=True)
top10 = sorted_tokens[:10]

print("Top-10 highest entropy tokens:")
print("-" * 70)
for rank, tok in enumerate(top10, 1):
    print(f"{rank:2d}. '{tok['token']}' "
          f"p.{tok['page']}:{tok['line']} "
          f"entropy={tok['entropy']:.4f}")

# Save top tokens
with open(OUTPUT_DIR / "E1_top_entropy_tokens.json", "w") as f:
    json.dump(top10, f, indent=2)
print("\nSaved to outputs/E1_top_entropy_tokens.json")

## Interpretation Guide

### Reading the waveform

- **Spikes**: Single high-entropy tokens amid lower surroundings → likely
  portmanteau words (condensation nodes) or proper names that carry multiple
  etymological layers simultaneously.

- **Plateaux**: Extended high-entropy regions → passages of maximum multilingual
  density (e.g. the thunder-words on p.4, the Anna Livia washerwomen chapter).

- **Valleys**: Low-entropy passages → the model has collapsed to a single
  interpretive frame; these are the baselines for E0 calibration.

### Expected findings

Based on Joyce scholarship and the McHugh annotations, we expect peaks at:

- **p. 4**: The 100-letter thunder-word (*bababadal…*) — maximum portmanteau
  condensation across 8+ languages.
- **p. 3, line 2**: *commodius vicus of recirculation* — Latin + Italian + English
  + Viconian philosophy in a single noun phrase.
- **p. 8**: *oystrygods gaggin fishygods* — Norse mythology + bodily comedy.

### Next steps

Feed the top-10 tokens identified here into **E2** for full superposition
discrimination across all 6 lenses.